<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Belgium: Regional Patent Distribution</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Map <strong>Belgian patent applicants across NUTS regions</strong> &mdash; their distribution, their evolution over time, and which of them are worth approaching for IP services.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; original notebook by <strong>Benoit</strong> (BE) &nbsp;&middot;&nbsp;
        <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">lifted to the TIP4PATLIBS method by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 660px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>Contents</strong>
            <br/>Setup &nbsp;&middot;&nbsp; Connect to PATSTAT
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Check the data edition
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Discover Belgium's NUTS-3 regions
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Set your region &amp; time window
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; <strong>Reality check</strong> &mdash; how much of Belgium is geolocatable
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>R&eacute;partition</strong> &mdash; companies &amp; families per region
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; <strong>Evolution</strong> &mdash; regional filing over time
            <br/>Step&nbsp;7 &nbsp;&middot;&nbsp; Axis 1 &mdash; portfolio depth (the ranked company list)
            <br/>Step&nbsp;8 &nbsp;&middot;&nbsp; Axis 2 &mdash; geographic reach
            <br/>Step&nbsp;9 &nbsp;&middot;&nbsp; <strong>Segment</strong> into lead tiers &amp; shortlist
            <br/>Step&nbsp;10 &nbsp;&middot;&nbsp; What this can &amp; cannot see
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 660px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            The default covers <strong>all of Belgium, 2015&ndash;2024</strong> and runs out of the box.
            To zoom into one region, change <code>NUTS_CODES</code> in Step&nbsp;3.
            Every query runs directly on PATSTAT inside EPO&nbsp;TIP &mdash; no BigQuery, no extra credentials.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025.
    </div>
</div>

## Setup: Connect to PATSTAT

Connects to PATSTAT on EPO TIP, imports Plotly for the charts, and defines a small
`run_query` helper that runs SQL and returns a pandas DataFrame with timing info.

In [ ]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import plotly.express as px
import time

# Connect to PATSTAT (PROD = the full production database on TIP)
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute SQL on PATSTAT and return a DataFrame with timing info."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    elapsed = time.time() - start
    df = pd.DataFrame(res)
    print(f"Query took {elapsed:.2f}s ({len(df)} rows)")
    return df

print("Connected to PATSTAT. Ready to run queries.")

---

## Step 1: Which data edition are we on?

Confirm how fresh the data is. On the current **Autumn 2025** edition the most recent
filing date should be around **2025-09-23**.

In [ ]:
df_edition = run_query("""
SELECT MAX(appln_filing_date) AS latest_filing_date
FROM tls201_appln
WHERE appln_filing_year BETWEEN 2024 AND 2026
""")
df_edition

---

## Step 2: Discover Belgium's NUTS-3 regions

We locate applicants by their **NUTS** region code. This lists every Belgian NUTS-3 code
actually present in the data, with its label &mdash; the arrondissements you will see in the
charts (e.g. `BE100` = Brussels, `BE211` = Antwerp, `BE332` = Li&egrave;ge).

We take **both geocoding vintages** with `nuts_level IN (3, 4)`: level 3 = EPO/ECOOM
geocoding (labelled), level 4 = OECD REGPAT (often unlabelled). Never filter on one alone.

In [ ]:
df_nuts = run_query("""
SELECT
    p.nuts,
    p.nuts_level,
    n.nuts_label,
    COUNT(DISTINCT p.person_id) AS person_records
FROM tls206_person p
LEFT JOIN tls904_nuts n ON n.nuts = p.nuts        -- LEFT JOIN: level-4 codes have no label
WHERE p.person_ctry_code = 'BE'
  AND p.nuts_level IN (3, 4)
GROUP BY p.nuts, p.nuts_level, n.nuts_label
ORDER BY p.nuts
""")
df_nuts

---

## Step 3: Set your region and time window

**The only cell you need to edit.** The default `['BE']` covers **all of Belgium**. To zoom
in, use a shorter prefix from Step 2 &mdash; `BE1` = Brussels-Capital, `BE2` = Flanders,
`BE3` = Wallonia, or a single arrondissement like `BE211` (Antwerp). Belgian codes are
stable across vintages, so a prefix is enough.

In [ ]:
# --- CHANGE THIS: your region and window --------------------------
NUTS_CODES = ['BE']        # whole Belgium; e.g. ['BE2'] = Flanders, ['BE211'] = Antwerp
YEAR_START = 2015
YEAR_END   = 2024
# ------------------------------------------------------------------

# Build the region filter: match any listed NUTS code as a prefix.
NUTS_WHERE = "(" + " OR ".join(f"p.nuts LIKE '{c}%'" for c in NUTS_CODES) + ")"

print("Region filter :", NUTS_WHERE)
print("Filing window :", YEAR_START, "-", YEAR_END)

---

## Step 4: Reality check &mdash; how much of Belgium is geolocatable?

Before mapping regions, see the coverage limit. NUTS codes are attached **only on the
EP/PCT route**. This splits Belgian *company* applicant records into those that resolve to a
**NUTS-3 region** vs those that carry **only the country** (`BE`). Expect the country-only
bucket to dominate &mdash; that is the national-only tail this method cannot place on the map.

In [ ]:
df_coverage = run_query(f"""
SELECT
    CASE WHEN p.nuts_level IN (3, 4) THEN 'NUTS-3 region'
         WHEN p.nuts_level = 0       THEN 'country only (BE)'
         ELSE 'other' END              AS geo_resolution,
    COUNT(DISTINCT p.person_id)        AS applicant_records
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
WHERE pa.applt_seq_nr > 0
  AND p.person_ctry_code = 'BE'
  AND p.psn_sector = 'COMPANY'
  AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
GROUP BY geo_resolution
ORDER BY applicant_records DESC
""")

total = df_coverage['applicant_records'].sum()
df_coverage['share_%'] = (100 * df_coverage['applicant_records'] / total).round(1)
df_coverage

---

## Step 5: R&eacute;partition &mdash; companies &amp; families per region

The core distribution: for every Belgian NUTS-3 region, how many **companies** file and how
many **patent families** they hold. We count *families* (`docdb_family_id`), not
applications, so one invention filed in ten countries counts once.

In [ ]:
df_region = run_query(f"""
SELECT
    p.nuts                            AS nuts_code,
    n.nuts_label,
    COUNT(DISTINCT p.han_name)        AS companies,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
LEFT JOIN tls904_nuts n   ON n.nuts = p.nuts
WHERE pa.applt_seq_nr > 0
  AND p.nuts_level IN (3, 4)
  AND {NUTS_WHERE}
  AND p.psn_sector = 'COMPANY'
  AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
GROUP BY p.nuts, n.nuts_label
ORDER BY families DESC
""")

# Use the code itself as a label where PATSTAT has none (level-4 records).
df_region['region'] = df_region['nuts_label'].fillna(df_region['nuts_code'])
df_region

Plot the families per region as a bar chart &mdash; the regional patenting profile of
Belgium at a glance.

In [ ]:
fig = px.bar(
    df_region.sort_values('families', ascending=False),
    x='region', y='families', color='families',
    title=f'Patent families per Belgian NUTS-3 region ({YEAR_START}–{YEAR_END})',
    labels={'region': 'NUTS-3 region', 'families': 'Patent families'},
)
fig.update_layout(height=700, xaxis_tickangle=-45, showlegend=False)
fig.show()

---

## Step 6: Evolution &mdash; regional filing over time

The same families broken down by **filing year**, as an animated bar-chart race. Missing
year-region combinations are filled with 0 so the animation stays stable frame to frame.

In [ ]:
df_evo = run_query(f"""
SELECT
    p.nuts                            AS nuts_code,
    n.nuts_label,
    a.appln_filing_year               AS filing_year,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
LEFT JOIN tls904_nuts n   ON n.nuts = p.nuts
WHERE pa.applt_seq_nr > 0
  AND p.nuts_level IN (3, 4)
  AND {NUTS_WHERE}
  AND p.psn_sector = 'COMPANY'
  AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
GROUP BY p.nuts, n.nuts_label, a.appln_filing_year
ORDER BY filing_year, families DESC
""")
df_evo['region'] = df_evo['nuts_label'].fillna(df_evo['nuts_code'])

# Complete the year x region grid with zeros for smooth animation frames.
years   = sorted(df_evo['filing_year'].unique())
regions = df_evo['region'].unique()
full = pd.MultiIndex.from_product([years, regions], names=['filing_year', 'region']).to_frame(index=False)
df_evo_full = full.merge(df_evo[['filing_year', 'region', 'families']],
                         on=['filing_year', 'region'], how='left').fillna({'families': 0})
print(f"{len(regions)} regions x {len(years)} years")
df_evo_full.head()

Animated horizontal bar-chart race &mdash; press play to watch each region's family count
grow year by year. Bars re-order so the leading regions rise to the top.

In [ ]:
fig = px.bar(
    df_evo_full,
    x='families', y='region', color='region',
    animation_frame='filing_year', animation_group='region',
    orientation='h',
    title='Bar-chart race — patent families per Belgian region by filing year',
    labels={'families': 'Patent families', 'region': 'NUTS-3 region', 'filing_year': 'Filing year'},
)
fig.update_layout(height=900, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.update_xaxes(range=[0, df_evo_full['families'].max() * 1.1])
fig.show()

---

## Step 7: Axis 1 &mdash; portfolio depth (the company list)

The ranked list of **company applicants in your region**, each with its number of **patent
families**. This is the core lead-generation deliverable. Corpus rules live in the `WHERE`
clause: applicants only, companies only, both NUTS vintages, families not filings.

In [ ]:
df_depth = run_query(f"""
SELECT
    p.han_name                        AS applicant,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
WHERE pa.applt_seq_nr > 0
  AND p.nuts_level IN (3, 4)
  AND {NUTS_WHERE}
  AND p.psn_sector = 'COMPANY'
  AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
GROUP BY applicant
ORDER BY families DESC
""")

print(f"{len(df_depth)} companies in scope")
df_depth.head(20)

---

## Step 8: Axis 2 &mdash; geographic reach

Depth is *how much* a company files; reach is *how far* it protects. For each company we
take **all members of each family** and flag which economic zones they reach beyond
Europe/PCT: North America (US, CA), Asia (CN, JP, KR, IN, TW, SG, IL), Oceania (AU, NZ).

In [ ]:
df_reach = run_query(f"""
WITH corp_fam AS (          -- one row per (company, family) in the region
  SELECT DISTINCT p.han_name AS applicant, a.docdb_family_id
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
),
fam_zone AS (               -- for each family: does ANY member reach each zone?
  SELECT cf.applicant, cf.docdb_family_id,
    MAX(CASE WHEN m.appln_auth IN ('US','CA')                          THEN 1 ELSE 0 END) AS z_na,
    MAX(CASE WHEN m.appln_auth IN ('CN','JP','KR','IN','TW','SG','IL') THEN 1 ELSE 0 END) AS z_asia,
    MAX(CASE WHEN m.appln_auth IN ('AU','NZ')                          THEN 1 ELSE 0 END) AS z_oce
  FROM corp_fam cf
  JOIN tls201_appln m ON m.docdb_family_id = cf.docdb_family_id
  GROUP BY cf.applicant, cf.docdb_family_id
)
SELECT
  applicant,
  COUNT(*)      AS families,
  SUM(z_na)     AS fam_north_america,
  SUM(z_asia)   AS fam_asia,
  SUM(z_oce)    AS fam_oceania
FROM fam_zone
GROUP BY applicant
ORDER BY families DESC
""")
df_reach.head(20)

---

## Step 9: Segment into lead tiers (depth &times; reach)

Combine both axes into a grid a PATLIB can act on. **Depth**: small (1&ndash;2 families),
medium (3&ndash;10), large (&gt;10). **Reach**: local (single route), regional (spans two
routes, stays in Europe/PCT), global (reaches beyond Europe). The grid counts companies per
tier.

In [ ]:
df_segments = run_query(f"""
WITH corp_fam AS (
  SELECT DISTINCT p.han_name AS applicant, a.docdb_family_id
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
),
fam_zone AS (
  SELECT cf.applicant, cf.docdb_family_id,
    MAX(CASE WHEN m.appln_auth = 'EP'                                  THEN 1 ELSE 0 END) AS z_ep,
    MAX(CASE WHEN m.appln_auth = 'WO'                                  THEN 1 ELSE 0 END) AS z_wo,
    MAX(CASE WHEN m.appln_auth IN ('US','CA')                          THEN 1 ELSE 0 END) AS z_na,
    MAX(CASE WHEN m.appln_auth IN ('CN','JP','KR','IN','TW','SG','IL') THEN 1 ELSE 0 END) AS z_asia,
    MAX(CASE WHEN m.appln_auth IN ('AU','NZ')                          THEN 1 ELSE 0 END) AS z_oce
  FROM corp_fam cf
  JOIN tls201_appln m ON m.docdb_family_id = cf.docdb_family_id
  GROUP BY cf.applicant, cf.docdb_family_id
),
company AS (
  SELECT applicant,
    COUNT(*)                                          AS families,
    MAX(z_ep + z_wo + z_na + z_asia + z_oce)          AS widest_family_zones,
    SUM(z_na) + SUM(z_asia) + SUM(z_oce)              AS beyond_europe_hits
  FROM fam_zone
  GROUP BY applicant
)
SELECT
  CASE WHEN families > 10 THEN 'large'
       WHEN families BETWEEN 3 AND 10 THEN 'medium'
       ELSE 'small' END                                   AS depth_tier,
  CASE WHEN beyond_europe_hits > 0 THEN 'global'
       WHEN widest_family_zones >= 2 THEN 'regional'
       ELSE 'local' END                                   AS reach_tier,
  COUNT(*)                                                AS companies
FROM company
GROUP BY depth_tier, reach_tier
ORDER BY depth_tier, reach_tier
""")

grid = (pd.pivot_table(df_segments, index='depth_tier', columns='reach_tier',
                       values='companies', aggfunc='sum', fill_value=0)
        .reindex(index=['small', 'medium', 'large'],
                 columns=['local', 'regional', 'global'], fill_value=0)
        .astype(int))
print("companies by depth (rows) x reach (cols) -- total:", int(df_segments['companies'].sum()))
grid

Turn the grid into the **named shortlist** you act on: one row per company, its family
count, and how many families reach each zone. Sort or filter to the tier you want to
approach.

In [ ]:
df_leads = df_reach.copy()
df_leads['depth_tier'] = pd.cut(df_leads['families'], bins=[0, 2, 10, 10**9],
                                labels=['small', 'medium', 'large'])
df_leads['beyond_europe'] = df_leads[['fam_north_america', 'fam_asia', 'fam_oceania']].sum(axis=1)
df_leads = df_leads.sort_values('families', ascending=False)

# Example: the internationally-active mid-to-large filers a PATLIB might approach first
# df_leads[(df_leads['depth_tier'] != 'small') & (df_leads['beyond_europe'] > 0)]
df_leads.head(20)

---

## Step 10: What this can &mdash; and cannot &mdash; see

<div style="background:#fffbeb; border:1px solid #fcd34d; border-radius:10px; padding:16px 20px;">
<strong>&#9888; A NUTS filter finds only the EP/PCT-active companies of a region.</strong>
<br/><br/>
As Step&nbsp;4 shows, NUTS region codes are attached <em>only</em> on the European/PCT route
(EPO at level&nbsp;3, OECD REGPAT at level&nbsp;4). A Belgian application filed <em>only</em>
nationally &mdash; never at the EPO or via PCT &mdash; carries <em>no</em> NUTS code and cannot
be placed on the map. Those are typically the smaller, locally-filing SMEs &mdash; exactly the
leads a PATLIB most wants to find.
<br/><br/>
So this analysis is the <strong>EP/PCT-active subset</strong> of Belgium: excellent for
internationally-minded filers, blind to the national-only tail. PATSTAT cannot recover them
by postcode (the structured ZIP field is empty and addresses are sparse).
</div>

**Closing the gap (Belgian national register).** The German TIP4PATLIBS material solves the
same problem with a *national-office* extension: pull the applicant's postcode from the
DPMA register, then map **postcode &rarr; NUTS-3 &rarr; NUTS-1** via the Eurostat GISCO
crosswalk, and union that with the PATSTAT-NUTS list. The Belgian equivalent would pair a
`pc2025_BE_NUTS-2024` crosswalk (free from Eurostat) with applicant addresses from the
Belgian national register &mdash; a natural next step, not built here.

**One caveat for a final ranking:** `han_name` can split one group across several rows.
Before publishing a league table, consolidate via `doc_std_name_id` / `psn_id`.

---
*Original notebook by Benoit (BE) &middot; lifted to the TIP4PATLIBS method &middot; Data: EPO PATSTAT Global, Autumn 2025 &middot; mtc.berlin*